<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook prepares QC tables for the joint long-read SV callsets. It downloads the sens_07 and sens_09 VCFs on T2T and GRCh38, relabels `SVTYPE=UNK` to `INV` (a Truvari annotation issue), and writes cohort-wide and per-sample SVTYPE and SVLEN counts consumed by `07_callset_qc_r.ipynb`.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# download merged callsets T2T
!gsutil cp {AOU_LR_DATA}/releases/sv_integration/T2T/v1/concat_annotated.sens_09.vcf.gz concat_annotated.sens_09.T2T.vcf.gz
!gsutil cp {AOU_LR_DATA}/releases/sv_integration/T2T/v1/concat_annotated.sens_07.vcf.gz concat_annotated.sens_07.T2T.vcf.gz

In [ ]:
# download merged callsets GRCh38
!gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_09.vcf.gz .
!gsutil cp {AOU_LR_DATA}/releases/sv_integration/GRCh38/v1/concat_annotated.sens_07.vcf.gz .

In [ ]:
# UNK-->INV for LR file (fixing known issue in truvari anno svinfo)
from pysam import VariantFile

def unk_to_inv(vcf_in_filename, vcf_out_filename):
    vcf_in = VariantFile(vcf_in_filename)
    vcf_out = VariantFile(vcf_out_filename, 'w', header=vcf_in.header)

    for rec in vcf_in.fetch():
        if rec.info['SVTYPE']=='UNK':
            rec.info['SVTYPE']='INV'
        vcf_out.write(rec) # write every variant

In [ ]:
# run UNK-->INV
unk_to_inv("concat_annotated.sens_09.vcf.gz","concat_annotated.sens_09.unk_to_inv.vcf")
!bcftools view -Oz concat_annotated.sens_09.unk_to_inv.vcf > concat_annotated.sens_09.unk_to_inv.vcf.gz
!tabix concat_annotated.sens_09.unk_to_inv.vcf.gz
!rm concat_annotated.sens_09.unk_to_inv.vcf

In [ ]:
# Whole-cohort SV counts by SVTYPE for each calibration sensitivity (sens_09 and the stricter sens_07).
!bcftools query -f "%SVTYPE\n" concat_annotated.sens_09.unk_to_inv.vcf.gz | sort \
    | uniq -c |awk 'OFS="\t" {print $2,$1}' > tpr09_whole_cohort_count_by_svtype

In [ ]:
!bcftools query -f "%SVTYPE\n" concat_annotated.sens_07.vcf.gz | sort \
    | uniq -c |awk 'OFS="\t" {print $2,$1}' > tpr07_whole_cohort_count_by_svtype

In [ ]:
# Small test region (chr1:1-5 Mbp) for trying the steps below quickly.
!bcftools view -Oz concat_annotated.sens_09.unk_to_inv.vcf.gz chr1:1000000-5000000 > test_small.vcf.gz

In [ ]:
# One row per SV with SVTYPE, SVLEN, AF, AC, and the comma-separated list of carrier samples (any
# non-reference genotype), for each callset and reference.
!bcftools query -f '%SVTYPE\t%SVLEN\t%AF\t%AC\t[%SAMPLE,]' -i 'GT=="alt"' \
    concat_annotated.sens_09.T2T.vcf.gz > concat_annotated.sens_09.T2T.tsv
!bcftools query -f '%SVTYPE\t%SVLEN\t%AF\t%AC\t[%SAMPLE,]' -i 'GT=="alt"' \
    concat_annotated.sens_07.T2T.vcf.gz > concat_annotated.sens_07.T2T.tsv
!bcftools query -f '%SVTYPE\t%SVLEN\t%AF\t%AC\t[%SAMPLE,]' -i 'GT=="alt"' \
    concat_annotated.sens_09.vcf.gz > concat_annotated.sens_09.tsv
!bcftools query -f '%SVTYPE\t%SVLEN\t%AF\t%AC\t[%SAMPLE,]' -i 'GT=="alt"' \
    concat_annotated.sens_07.vcf.gz > concat_annotated.sens_07.tsv

In [ ]:
# output variants per sample by SVTYPE

infile=open("concat_annotated.sens_09.tsv",'r')
outfile = open("concat_annotated.sens_09.variants_per_sample_by_SVTYPE",'w')
SVTYPES = ['INS','DEL','INV']

sample_counts_per_var_type = {} # sample-->SVTYPE-->count

for line in infile:
    line=line.strip().split('\t')
    SVTYPE=line[0]
    SVLEN=int(line[1])
    samples = line[4].rstrip(',').split(',')
    for sample in samples:
        if sample not in sample_counts_per_var_type:
            sample_counts_per_var_type[sample] = {}
            for type_init in SVTYPES:
                sample_counts_per_var_type[sample][type_init] = 0
            sample_counts_per_var_type[sample][SVTYPE] = 1
        else:
            sample_counts_per_var_type[sample][SVTYPE] += 1
                
for sample in sample_counts_per_var_type:
    total = 0
    for SVTYPE in SVTYPES:
        total+=sample_counts_per_var_type[sample][SVTYPE]
    outfile.write("%s\t%d" % (sample, total))
    for SVTYPE in SVTYPES:
        outfile.write("\t%d" % (sample_counts_per_var_type[sample][SVTYPE]))
    outfile.write("\n")

infile.close()
outfile.close()

In [ ]:
# SVTYPE and SVLEN of every SV, for the length-distribution plots in 07_callset_qc_r.
!bcftools query -f '%SVTYPE\t%SVLEN\n' concat_annotated.sens_09.unk_to_inv.vcf.gz \
    > tpr09_whole_cohort_length_by_svtype